In [0]:
# Instalar biblioteca para descompactar 7z
%pip install py7zr

# Conexão FTP e download
import ftplib
import os

# Diretório temporário no Databricks
temp_dir = '/tmp/caged_data'
os.makedirs(temp_dir, exist_ok=True)

ftp = ftplib.FTP('ftp.mtps.gov.br')
ftp.login()
ftp.cwd('pdet/microdados/NOVO CAGED/2026/202602')

# Download arquivo
local_file = f'{temp_dir}/CAGEDMOV202602.7z'
with open(local_file, 'wb') as f:
    ftp.retrbinary('RETR CAGEDMOV202602.7z', f.write)

ftp.quit()
print(f"Arquivo baixado em: {local_file}")

In [0]:
# Extração do arquivo 7z
import py7zr
import os

temp_dir = '/tmp/caged_data'
extract_dir = f'{temp_dir}/extracted'
os.makedirs(extract_dir, exist_ok=True)

with py7zr.SevenZipFile(f'{temp_dir}/CAGEDMOV202602.7z', mode='r') as archive:
    archive.extractall(extract_dir)

print(f"Arquivo extraído em: {extract_dir}")

In [0]:
# Detectar encoding do arquivo CSV
import chardet

csv_path = f'{extract_dir}/CAGEDMOV202602.txt'

with open(csv_path, 'rb') as f:
    result = chardet.detect(f.read(100000))

print(f"Encoding detectado: {result['encoding']} (confiança: {result['confidence']})")
encoding = result['encoding']

In [0]:
# Ler o CSV usando pandas primeiro (serverless não permite Spark acessar /tmp/)
import pandas as pd
pandas_df = pd.read_csv(csv_path, delimiter=';', encoding=encoding)

# Converter para Spark DataFrame
df = spark.createDataFrame(pandas_df)

print(f"Total de registros: {df.count():,}")
print(f"\nSchema:")
df.printSchema()
print(f"\nAmostra dos dados:")
display(df.limit(10))

In [0]:
# Definir o nome da tabela (ajuste catalog e schema conforme sua estrutura)
catalog = 'workspace'  # Ajuste para seu catalog
schema = 'default'  # Ajuste para seu schema
table_name = 'caged_movimentacoes_202602'

# Salvar como Delta Table no Unity Catalog
df.write.format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable(f'{catalog}.{schema}.{table_name}')

print(f"✓ Delta Table criada: {catalog}.{schema}.{table_name}")
print(f"Total de registros salvos: {df.count():,}")